# Target-Supervised Contrastive Embedding (with SSL regularizer)

Trains the encoder with a **combined contrastive objective**:

```
loss = ssl_loss + 0.2 * supcon_loss
```

- **`ssl_loss`** (NT-Xent / SimCLR) -- *unsupervised*: positives are only
  same-customer cross-view pairs; every other sample is a negative. Shapes
  the general geometry of the embedding without using labels.
- **`supcon_loss`** (`ts_embed.loss.SupConLoss`) -- *supervised* + class
  balanced: same-label customers are *also* positives, so the embedding
  becomes target-aligned. A `WeightedRandomSampler` balances batches.

The two losses share the SupCon projection head: one forward through the
projector per step, two different positive-masks. The combined objective is
usually more robust than either alone -- the SSL term provides task-agnostic
regularization that keeps the embedding from overfitting noisy supervised
positives, while the SupCon term tethers the geometry to the target.

Pipeline:
1. Synthetic series + a binary target driven by latent risk factors.
2. Encoder + `SupConLoss` + class-balanced batches.
3. Train with the combined loss; per-epoch holdout monitoring of both terms.
4. Evaluate target-discriminability of the learned embedding.

In [ ]:
import sys, pathlib
REPO_ROOT = pathlib.Path.cwd()
if not (REPO_ROOT / 'ts_embed').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from ts_embed.data import (DatasetPaths, TimeSeriesDataset,
                           TimeFeatureMasker, ContrastiveCollator)
from ts_embed.model import TSEncoder, TSEncoderConfig
from ts_embed.loss import SupConLoss, supcon_loss

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 1. Synthetic data + binary target (already split into train / holdout)

In production you start with the two splits already separated: each split has
its own input files and its own target file. We mirror that here by writing
the synthetic data to two directories:

```
data_demo/train/    numeric.npy, missing.npy, categorical.npy, target.npy
data_demo/holdout/  numeric.npy, missing.npy, categorical.npy, target.npy
```

Three latent factors (`delinq`, `payment`, `balance`) drive feature blocks
and also a binary target -- the target is recoverable from the series.

In [ ]:
N, T, F_NUM, F_CAT = 6000, 24, 98, 2
N_TRAIN = int(0.85 * N); N_HOLDOUT = N - N_TRAIN
rng = np.random.default_rng(0)

f_delinq  = rng.standard_normal(N).astype(np.float32)
f_payment = rng.standard_normal(N).astype(np.float32)
f_balance = rng.standard_normal(N).astype(np.float32)

numeric = (0.3 * rng.standard_normal((N, T, F_NUM))).astype(np.float32)
t_axis = np.linspace(0, 1, T, dtype=np.float32)[None, :, None]
numeric[:, :, 0:33]  += f_balance[:, None, None] * (0.5 + t_axis)
numeric[:, :, 33:66] += f_payment[:, None, None] + 0.3 * rng.standard_normal(
    (N, T, 33)).astype(np.float32)
numeric[:, :, 66:98] += f_delinq[:, None, None] * t_axis * 1.5

missing = (rng.random((N, T, F_NUM)) < 0.1).astype(np.uint8)
feat_mean = numeric.mean((0, 1), keepdims=True)
numeric = np.where(missing == 1, feat_mean, numeric).astype(np.float32)
categorical = np.stack([
    (rng.random((N, T)) < 0.5).astype(np.int8),
    (rng.random((N, T)) < 0.5).astype(np.int8),
], axis=-1)

logit = 1.3 * f_delinq - 1.0 * f_payment + 0.6 * f_balance - 1.4
prob = 1.0 / (1.0 + np.exp(-logit))
target = (rng.random(N) < prob).astype(np.int64)

# Split into train and holdout, save each to its own directory. In production
# you would already have these as separate files — this just simulates that.
perm = rng.permutation(N)
train_rows = perm[:N_TRAIN]; holdout_rows = perm[N_TRAIN:]

def _save(out_dir, rows):
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / 'numeric.npy',     numeric[rows])
    np.save(out_dir / 'missing.npy',     missing[rows])
    np.save(out_dir / 'categorical.npy', categorical[rows])
    np.save(out_dir / 'target.npy',      target[rows])

data_dir = REPO_ROOT / 'data_demo'
_save(data_dir / 'train',   train_rows)
_save(data_dir / 'holdout', holdout_rows)
print(f'train={N_TRAIN}  holdout={N_HOLDOUT}  '
      f'train pos rate={target[train_rows].mean():.3f}  '
      f'holdout pos rate={target[holdout_rows].mean():.3f}')

## 2. Encoder + SupConLoss + class-balanced loader

We open the two splits as **separate** `TimeSeriesDataset` instances and wrap
each in a tiny `TargetDataset` so every item yields its target alongside the
feature dict -- no index bookkeeping or custom collator class is needed.
`SupConLoss` carries its own projection head; the deliverable embedding is
the **encoder output**, not the projection.

With an imbalanced target we use a `WeightedRandomSampler` so each batch has
enough same-(minority)-label pairs for the supervised positives to be
meaningful. The unsupervised NT-Xent term (added in the training loop)
reuses the SupCon projection head so both losses operate on a single
projection per batch.

In [ ]:
train_paths = DatasetPaths(
    numeric=data_dir / 'train' / 'numeric.npy',
    missing=data_dir / 'train' / 'missing.npy',
    categorical=data_dir / 'train' / 'categorical.npy')
holdout_paths = DatasetPaths(
    numeric=data_dir / 'holdout' / 'numeric.npy',
    missing=data_dir / 'holdout' / 'missing.npy',
    categorical=data_dir / 'holdout' / 'categorical.npy')

train_base   = TimeSeriesDataset(train_paths)
holdout_base = TimeSeriesDataset(holdout_paths)

target_train   = torch.from_numpy(np.load(data_dir / 'train'   / 'target.npy')).long()
target_holdout = torch.from_numpy(np.load(data_dir / 'holdout' / 'target.npy')).long()


class TargetDataset(Dataset):
    """Wraps a base dataset so each item yields its target alongside the
    feature dict."""
    def __init__(self, base, targets):
        assert len(base) == len(targets), 'base and targets length mismatch'
        self.base = base; self.targets = targets

    def __len__(self):
        return len(self.base)

    def __getitem__(self, i):
        item = self.base[i]
        item['target'] = self.targets[i]
        return item


train_ds   = TargetDataset(train_base,   target_train)
holdout_ds = TargetDataset(holdout_base, target_holdout)

masker = TimeFeatureMasker(time_mask_prob=0.25, feature_mask_prob=0.30,
                           n_time_spans=2, feat_span_frac=0.5)
_cc = ContrastiveCollator(masker)

def collate(batch):
    """Build the two contrastive views and stack the per-row targets."""
    targets = torch.stack([b.pop('target') for b in batch])
    out = _cc(batch)
    out['target'] = targets
    return out

# Inverse-frequency weights over the train split for class-balanced batches.
cls_count = torch.bincount(target_train, minlength=2).float()
sample_w = (1.0 / cls_count)[target_train]
sampler = WeightedRandomSampler(sample_w, num_samples=len(train_ds), replacement=True)

loader = DataLoader(train_ds, batch_size=256, sampler=sampler, drop_last=True,
                    num_workers=0, collate_fn=collate)

cfg = TSEncoderConfig(n_numeric=F_NUM, cat_cardinalities=(2, 2), seq_len=T,
                      d_model=128, n_layers=3, n_heads=4, proj_dim=128)
encoder = TSEncoder(cfg).to(device)
supcon = SupConLoss(in_dim=cfg.d_model, proj_dim=128, proj_hidden=256,
                    temperature=0.1).to(device)
params = list(encoder.parameters()) + list(supcon.parameters())
optim = torch.optim.AdamW(params, lr=1e-3, weight_decay=1e-4)
print(f'train n={len(train_ds)} pos rate={float(target_train.float().mean()):.3f} | '
      f'holdout n={len(holdout_ds)} pos rate={float(target_holdout.float().mean()):.3f}')

## 3. Training loop with holdout-vintage monitoring

Each step:

1. Encode both masked / unmasked views.
2. Compute **both** contrastive losses through a SHARED projection -- NT-Xent
   (`ssl`) and class-balanced SupCon (`sup`).
3. Backprop the combined loss: `SSL_WEIGHT * ssl + SUPCON_WEIGHT * sup`.

The two losses differ only in their positive masks (`ssl` uses only
same-customer cross-view pairs; `sup` adds same-label cross-customer pairs),
so we project the embeddings *once* and reuse the `(2B, proj_dim)` features.

Every epoch we report the **per-component holdout losses** plus the
**linear-probe AUC**. The holdout pass reseeds the RNG so the random masking
augmentation is identical across epochs (the curves are directly comparable);
the RNG state is restored afterwards so training is unaffected.

In [ ]:
# Loss weighting per spec: total = ssl + 0.2 * sup.
SSL_WEIGHT, SUPCON_WEIGHT = 1.0, 0.2


def dual_contrastive_loss(emb_a, emb_b, target, supcon_module):
    """NT-Xent (ssl) + class-balanced SupCon (sup) via the SHARED projector.

    NT-Xent: positives are only same-customer cross-view pairs.
    SupCon:  NT-Xent positives PLUS same-label cross-customer pairs.

    Both losses operate on the same `(2B, proj_dim)` features so the
    projection is computed exactly once per training step.
    """
    b = emb_a.size(0)
    fa = torch.nn.functional.normalize(supcon_module.proj(emb_a), dim=-1)
    fb = torch.nn.functional.normalize(supcon_module.proj(emb_b), dim=-1)
    feats = torch.cat([fa, fb], dim=0)
    dev = feats.device
    idx = torch.arange(b, device=dev)

    # NT-Xent positive mask: only diagonal cross-view pairs are True.
    pm_ssl = torch.zeros(2 * b, 2 * b, dtype=torch.bool, device=dev)
    pm_ssl[idx, idx + b] = True
    pm_ssl[idx + b, idx] = True
    ssl = supcon_loss(feats, pm_ssl, supcon_module.temperature)

    # SupCon positive mask: same-label pairs tiled into (2B, 2B) + cross-view.
    base = target.unsqueeze(0) == target.unsqueeze(1)               # (B, B)
    pm_sup = torch.cat(
        [torch.cat([base, base], dim=1), torch.cat([base, base], dim=1)], dim=0)
    pm_sup[idx, idx + b] = True
    pm_sup[idx + b, idx] = True
    sup = supcon_loss(feats, pm_sup, supcon_module.temperature)
    return ssl, sup


def roc_auc(y, s):
    y = np.asarray(y); s = np.asarray(s)
    try:
        from sklearn.metrics import roc_auc_score
        return float(roc_auc_score(y, s))
    except ImportError:
        order = np.argsort(s, kind='mergesort')
        ranks = np.empty(len(s), float)
        sr = s[order]; i = 0
        while i < len(sr):
            j = i
            while j + 1 < len(sr) and sr[j + 1] == sr[i]:
                j += 1
            ranks[order[i:j + 1]] = 0.5 * (i + j) + 1.0
            i = j + 1
        pos = y == 1
        npos = int(pos.sum()); nneg = len(y) - npos
        if npos == 0 or nneg == 0:
            return float('nan')
        return float((ranks[pos].sum() - npos * (npos + 1) / 2) / (npos * nneg))


@torch.no_grad()
def embed_dataset(ts_dataset):
    encoder.eval()
    n_rows = int(len(ts_dataset))
    Z = []
    for start in range(0, n_rows, 512):
        end = start + 512 if start + 512 < n_rows else n_rows
        sl = slice(start, end)
        num = torch.from_numpy(np.asarray(ts_dataset.numeric[sl], dtype=np.float32)).to(device)
        mis = torch.from_numpy(np.asarray(ts_dataset.missing[sl], dtype=np.float32)).to(device)
        cat = torch.from_numpy(np.asarray(ts_dataset.categorical[sl], dtype=np.int64)).to(device)
        Z.append(encoder(num, mis, cat).float().cpu().numpy())
    return np.concatenate(Z)


def probe_auc():
    Ztr = embed_dataset(train_base); Zho = embed_dataset(holdout_base)
    try:
        from sklearn.linear_model import LogisticRegression
        lp = LogisticRegression(max_iter=2000, class_weight='balanced')
        lp.fit(Ztr, target_train.numpy())
        return roc_auc(target_holdout.numpy(), lp.predict_proba(Zho)[:, 1])
    except ImportError:
        return float('nan')


# Deterministic holdout loader: no sampler, no drop_last, sequential.
holdout_loader = DataLoader(holdout_ds, batch_size=256, shuffle=False,
                            drop_last=False, num_workers=0, collate_fn=collate)


@torch.no_grad()
def holdout_loss():
    """Per-component contrastive losses on the holdout vintage. Returns a
    dict with 'ssl', 'sup', and the combined 'loss'. RNG is reseeded so the
    random masking is reproducible across epochs."""
    encoder.eval(); supcon.eval()
    rng_state = torch.get_rng_state(); torch.manual_seed(123)
    try:
        sums = {'ssl': 0.0, 'sup': 0.0}; n_seen = 0
        for batch in holdout_loader:
            num_a = batch['numeric_a'].to(device); mis_a = batch['missing_a'].to(device)
            keep_a = batch['time_keep_a'].to(device)
            num_b = batch['numeric_b'].to(device); mis_b = batch['missing_b'].to(device)
            keep_b = batch['time_keep_b'].to(device)
            cat_a = batch['categorical_a'].to(device)
            cat_b = batch['categorical_b'].to(device)
            tgt = batch['target'].to(device)
            ea = encoder(num_a, mis_a, cat_a, keep_a)
            eb = encoder(num_b, mis_b, cat_b, keep_b)
            ssl, sup = dual_contrastive_loss(ea, eb, tgt, supcon)
            bs = num_a.size(0)
            sums['ssl'] += float(ssl) * bs
            sums['sup'] += float(sup) * bs
            n_seen += bs
    finally:
        torch.set_rng_state(rng_state)
    ssl_avg = sums['ssl'] / max(n_seen, 1)
    sup_avg = sums['sup'] / max(n_seen, 1)
    return {'ssl': ssl_avg, 'sup': sup_avg,
            'loss': SSL_WEIGHT * ssl_avg + SUPCON_WEIGHT * sup_avg}


EPOCHS = 15
history = []
for epoch in range(EPOCHS):
    encoder.train(); supcon.train()
    sums = {'ssl': 0.0, 'sup': 0.0, 'loss': 0.0}; n_seen = 0
    for batch in loader:
        num_a = batch['numeric_a'].to(device); mis_a = batch['missing_a'].to(device)
        keep_a = batch['time_keep_a'].to(device)
        num_b = batch['numeric_b'].to(device); mis_b = batch['missing_b'].to(device)
        keep_b = batch['time_keep_b'].to(device)
        cat_a = batch['categorical_a'].to(device)
        cat_b = batch['categorical_b'].to(device)
        tgt = batch['target'].to(device)

        emb_a = encoder(num_a, mis_a, cat_a, keep_a)
        emb_b = encoder(num_b, mis_b, cat_b, keep_b)
        ssl, sup = dual_contrastive_loss(emb_a, emb_b, tgt, supcon)
        loss = SSL_WEIGHT * ssl + SUPCON_WEIGHT * sup

        optim.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        optim.step()
        bs = num_a.size(0)
        sums['ssl'] += float(ssl) * bs
        sums['sup'] += float(sup) * bs
        sums['loss'] += float(loss) * bs
        n_seen += bs

    train_stats = {k: v / max(n_seen, 1) for k, v in sums.items()}
    h = holdout_loss()
    auc = probe_auc()
    history.append({'epoch': epoch, 'train': train_stats, 'holdout': h, 'auc': auc})

    print(f"epoch {epoch:2d}  "
          f"train_loss={train_stats['loss']:.3f} "
          f"(ssl={train_stats['ssl']:.3f}  sup={train_stats['sup']:.3f})  "
          f"holdout_loss={h['loss']:.3f} "
          f"(ssl={h['ssl']:.3f}  sup={h['sup']:.3f})  "
          f"AUC={auc:.4f}")

## 4. Is the embedding target-aligned?

Three checks on the held-out split:
1. **Linear-probe AUC** — logistic regression on the embedding.
2. **kNN AUC** — cosine-kNN vote; SupCon optimizes exactly this geometry.
3. **Collapse guard** — per-dimension std must stay > 0.

In [ ]:
Ztr = embed_dataset(train_base)
Zte = embed_dataset(holdout_base)
ytr = target_train.numpy()
yte = target_holdout.numpy()

# 1. linear probe — fit on train embedding, evaluate on holdout
try:
    from sklearn.linear_model import LogisticRegression
    lp = LogisticRegression(max_iter=2000, class_weight='balanced').fit(Ztr, ytr)
    lin_auc = roc_auc(yte, lp.predict_proba(Zte)[:, 1])
except ImportError:
    lin_auc = float('nan')

# 2. cosine kNN vote — neighbours from the train split
def knn_score(Xtr, ytr, Xte, k=25):
    Xtr = Xtr / (np.linalg.norm(Xtr, axis=1, keepdims=True) + 1e-8)
    Xte = Xte / (np.linalg.norm(Xte, axis=1, keepdims=True) + 1e-8)
    sims = Xte @ Xtr.T
    nn = np.argpartition(-sims, k, axis=1)[:, :k]
    return ytr[nn].mean(axis=1)  # fraction of positive neighbours

knn_auc = roc_auc(yte, knn_score(Ztr, ytr, Zte))

print(f'held-out linear-probe AUC : {lin_auc:.4f}')
print(f'held-out kNN AUC          : {knn_auc:.4f}')

# Collapse guard on the train embedding.
std = Ztr.std(0)
print(f'embedding per-dim std: mean={std.mean():.3f} min={std.min():.3f}')
assert std.mean() > 1e-2, 'COLLAPSE: embedding nearly constant'

# Class-mean separation on the train split.
mu_pos = Ztr[ytr == 1].mean(0); mu_neg = Ztr[ytr == 0].mean(0)
between = np.linalg.norm(mu_pos - mu_neg)
within = np.linalg.norm(Ztr - np.where(ytr[:, None] == 1, mu_pos, mu_neg),
                       axis=1).mean()
print(f'class-mean separation ratio = {between / within:.2f} '
      f'(higher = more target-aligned)')

## Next steps

- **Downstream use**: this embedding is target-aligned — feed it to your
  XGBoost (ideally *concatenated* with existing aggregated features, not
  replacing them) and check grouped feature importance / SHAP.
- **Imbalance**: the `WeightedRandomSampler` here balances batches; for very
  rare targets also consider a larger batch size so each batch holds enough
  minority positives.
- **Temperature**: lower `temperature` (0.05-0.1) sharpens the contrast; tune
  on the held-out probe AUC.
- **Combine objectives**: SupCon can be added to the self-supervised VICReg /
  structured losses (weighted sum) so the embedding keeps general structure
  while gaining target alignment.
- **Scale-out**: for DDP, gather projections across ranks (mirror VICReg's
  `_maybe_all_gather`) so SupCon sees more negatives per step.